# Algoritmos básicos · Números especiales y sucesiones

Pequeños algoritmos sobre las cifras y la aritmética de los enteros: dígitos de control, familias de números con nombre propio (felices, Lychrel, abundantes, Harshad…), huecos entre primos, ternas pitagóricas, sucesiones definidas por recurrencia, el problema de Josefo, la secuencia de Farey, combinatoria sin factoriales, operaciones de bits y los dígitos de π con enteros grandes. Cada resultado se comprueba con fuerza bruta, la biblioteca estándar u otra forma de calcularlo. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python) y de [Edabit](https://edabit.com/es/challenges/python3).

In [1]:
import math
import random
from collections import Counter, deque
from fractions import Fraction
from itertools import pairwise, permutations


def cifras(n):
    return [int(c) for c in str(n)]


def es_primo(n):
    if n < 2:
        return False
    return all(n % d for d in range(2, math.isqrt(n) + 1))

## 1. Algoritmo de Luhn

Las tarjetas y muchos identificadores llevan un dígito de control calculado con el algoritmo de Luhn: se recorren las cifras de derecha a izquierda, se dobla una sí y otra no empezando por la segunda, a cada cifra doblada mayor que 9 se le resta 9 y el número es válido si la suma es múltiplo de 10. Escribir una función que muestre los pasos en una tabla, otra que valide y otra que calcule el dígito de control que hay que añadir a un número. Comprobar con números aleatorios que los números generados son válidos, que cambiar cualquier cifra los invalida y qué intercambios de dos cifras vecinas no detecta.

In [2]:
def pasos_luhn(numero):
    filas = []
    for i, d in enumerate(reversed(numero)):
        d = int(d)
        doblado = 2 * d if i % 2 else d
        filas.append((i, d, doblado, doblado - 9 if doblado > 9 else doblado))
    return filas


def es_valido_luhn(numero):
    return sum(f[3] for f in pasos_luhn(numero)) % 10 == 0


def digito_control(parcial):
    # Con un 0 al final, las cifras de parcial quedan en la paridad que tendrán
    total = sum(f[3] for f in pasos_luhn(parcial + "0"))
    return str(-total % 10)


numero = "40719302"
print(f"{'pos':>3} {'cifra':>5} {'doblada':>7} {'suma':>4}")
for i, d, doblado, valor in pasos_luhn(numero):
    print(f"{i:>3} {d:>5} {doblado:>7} {valor:>4}")
total = sum(f[3] for f in pasos_luhn(numero))
print(f"Total {total}: {'válido' if es_valido_luhn(numero) else 'no válido'}")
control = digito_control(numero)
print(f"Dígito de control para {numero}: {control} → {numero + control} es válido: {es_valido_luhn(numero + control)}")

pos cifra doblada suma
  0     2       2    2
  1     0       0    0
  2     3       3    3
  3     9      18    9
  4     1       1    1
  5     7      14    5
  6     0       0    0
  7     4       8    8
Total 28: no válido
Dígito de control para 40719302: 8 → 407193028 es válido: True


In [3]:
random.seed(11)
generados = []
for _ in range(500):
    base = "".join(random.choice("0123456789") for _ in range(15))
    generados.append(base + digito_control(base))
assert all(es_valido_luhn(n) for n in generados)

for n in generados[:100]:
    for i, c in enumerate(n):
        for otra in "0123456789".replace(c, ""):
            assert not es_valido_luhn(n[:i] + otra + n[i + 1:])

no_detectados = Counter()
for n in generados:
    for i in range(len(n) - 1):
        if n[i] != n[i + 1]:
            cambiado = n[:i] + n[i + 1] + n[i] + n[i + 2:]
            if es_valido_luhn(cambiado):
                no_detectados[n[i : i + 2]] += 1
print("Cambiar una cifra: siempre se detecta")
print("Intercambios de vecinas no detectados:", dict(no_detectados))
assert set(no_detectados) <= {"09", "90"}

Cambiar una cifra: siempre se detecta
Intercambios de vecinas no detectados: {'09': 82, '90': 73}


## 2. Tabla de predicados sobre las cifras

Escribir un predicado para cada familia y mostrar una tabla que diga a cuáles pertenece cada número de una lista:

- **Harshad**: divisible por la suma de sus cifras.
- **Disarium**: igual a la suma de sus cifras elevadas a su posición ($135 = 1^1 + 3^2 + 5^3$).
- **Estético**: de dos o más cifras, y dos cifras vecinas se diferencian siempre en 1 (`2123`).
- **Repdígito**: de dos o más cifras, todas iguales.
- **Pandigital**: usa las cifras del 1 al 9 una vez cada una.
- **Semiprimo**: producto de dos primos, no necesariamente distintos.

Comprobar cada predicado con otra forma de obtener la familia: los Disarium menores que $10^4$, los estéticos generándolos cifra a cifra, los repdígitos con la fórmula $d \cdot \frac{10^k - 1}{9}$ y los semiprimos multiplicando parejas de primos.

In [4]:
def es_harshad(n):
    return n % sum(cifras(n)) == 0


def es_disarium(n):
    return n == sum(d**i for i, d in enumerate(cifras(n), start=1))


def es_estetico(n):
    c = cifras(n)
    return len(c) > 1 and all(abs(a - b) == 1 for a, b in pairwise(c))


def es_repdigito(n):
    return n > 9 and len(set(str(n))) == 1


def es_pandigital(n):
    return sorted(str(n)) == list("123456789")


def menor_factor(n):
    return next((d for d in range(2, math.isqrt(n) + 1) if n % d == 0), n)


def es_semiprimo(n):
    if n < 4:
        return False
    p = menor_factor(n)
    return p < n and es_primo(n // p)


predicados = {
    "Harshad": es_harshad, "Disarium": es_disarium, "estético": es_estetico,
    "repdígito": es_repdigito, "pandigital": es_pandigital, "semiprimo": es_semiprimo,
}
numeros = [21, 49, 89, 111, 135, 175, 343, 1306, 2123, 4444, 6789, 98765, 123456789, 381654729]
print(f"{'n':>10}" + "".join(f"{nombre:>11}" for nombre in predicados))
for n in numeros:
    print(f"{n:>10}" + "".join(f"{'✓' if p(n) else '·':>11}" for p in predicados.values()))

         n    Harshad   Disarium   estético  repdígito pandigital  semiprimo
        21          ✓          ·          ✓          ·          ·          ✓
        49          ·          ·          ·          ·          ·          ✓
        89          ·          ✓          ✓          ·          ·          ·
       111          ✓          ·          ·          ✓          ·          ✓
       135          ✓          ✓          ·          ·          ·          ·
       175          ·          ✓          ·          ·          ·          ·
       343          ·          ·          ✓          ·          ·          ·
      1306          ·          ✓          ·          ·          ·          ✓
      2123          ·          ·          ✓          ·          ·          ✓
      4444          ·          ·          ·          ✓          ·          ·
      6789          ·          ·          ✓          ·          ·          ·
     98765          ·          ·          ✓          ·          ·          ✓

In [5]:
assert [n for n in range(1, 10_000) if es_disarium(n)] == [*range(1, 10), 89, 135, 175, 518, 598, 1306, 1676, 2427]

esteticos, frontera = set(), list(range(1, 10))
while frontera:
    n = frontera.pop()
    for d in (n % 10 - 1, n % 10 + 1):
        if 0 <= d <= 9 and (m := 10 * n + d) < 100_000:
            esteticos.add(m)
            frontera.append(m)
assert esteticos == {n for n in range(100_000) if es_estetico(n)}

repdigitos = {d * (10**k - 1) // 9 for d in range(1, 10) for k in range(2, 7)}
assert repdigitos == {n for n in range(1_000_000) if es_repdigito(n)}

primos = [p for p in range(2, 500) if es_primo(p)]
semiprimos = {p * q for p in primos for q in primos if p <= q and p * q < 1000}
assert semiprimos == {n for n in range(1000) if es_semiprimo(n)}

assert sum(es_harshad(n) for n in range(1, 101)) == 33
print(f"Estéticos < 10⁵: {len(esteticos)}; repdígitos < 10⁶: {len(repdigitos)}; semiprimos < 1000: {len(semiprimos)}")

Estéticos < 10⁵: 226; repdígitos < 10⁶: 45; semiprimos < 1000: 299


## 3. Números felices

Un número es feliz si, al sustituirlo una y otra vez por la suma de los cuadrados de sus cifras, se llega a 1 ($19 \to 82 \to 68 \to 100 \to 1$). Si no, la sucesión entra en un ciclo que no contiene el 1. Detectar el ciclo de dos formas: guardando los valores vistos en un conjunto y con el algoritmo de Floyd (liebre y tortuga), que no necesita memoria extra. Comprobar que coinciden hasta 10 000, listar los felices hasta 100 y mostrar el ciclo en el que caen todos los infelices.

In [6]:
def siguiente_feliz(n):
    return sum(d * d for d in cifras(n))


def es_feliz_conjunto(n):
    vistos = set()
    while n != 1 and n not in vistos:
        vistos.add(n)
        n = siguiente_feliz(n)
    return n == 1


def es_feliz_floyd(n):
    tortuga, liebre = n, siguiente_feliz(n)
    while liebre != 1 and tortuga != liebre:
        tortuga = siguiente_feliz(tortuga)
        liebre = siguiente_feliz(siguiente_feliz(liebre))
    return liebre == 1


assert all(es_feliz_conjunto(n) == es_feliz_floyd(n) for n in range(1, 10_001))
print("Felices hasta 100:", [n for n in range(1, 101) if es_feliz_floyd(n)])

ciclo, n = [], 4
while n not in ciclo:
    ciclo.append(n)
    n = siguiente_feliz(n)
print("Ciclo de los infelices:", " → ".join(map(str, ciclo + [ciclo[0]])))


def llega_al_ciclo(n):
    for _ in range(100):
        if n in ciclo:
            return True
        n = siguiente_feliz(n)
    return False


assert all(llega_al_ciclo(n) for n in range(1, 10_001) if not es_feliz_floyd(n))

Felices hasta 100: [1, 7, 10, 13, 19, 23, 28, 31, 32, 44, 49, 68, 70, 79, 82, 86, 91, 94, 97, 100]
Ciclo de los infelices: 4 → 16 → 37 → 58 → 89 → 145 → 42 → 20 → 4


## 4. Números de Lychrel

Al sumar a un número su inverso ($56 + 65 = 121$) y repetir, casi siempre se llega pronto a un capicúa. Llamar **orden** de un número al número de pasos que necesita (al menos uno). Calcular el orden de algunos números, el menor número de cada orden entre los menores que 1000, y los candidatos a número de Lychrel menores que 1000: los que no llegan a un capicúa en 500 pasos. Seguir 196 durante 1000 pasos y ver cuántas cifras alcanza sin encontrar ningún capicúa.

In [7]:
def invierte_y_suma(n):
    return n + int(str(n)[::-1])


def es_capicua(n):
    return str(n) == str(n)[::-1]


def orden(n, limite=500):
    for paso in range(1, limite + 1):
        n = invierte_y_suma(n)
        if es_capicua(n):
            return paso, n
    return None, n


for n in (56, 57, 59, 89, 10_911):
    pasos, capicua = orden(n)
    print(f"{n:>6}: {pasos:>2} pasos → {capicua}")

    56:  1 pasos → 121
    57:  2 pasos → 363
    59:  3 pasos → 1111
    89: 24 pasos → 8813200023188
 10911: 55 pasos → 4668731596684224866951378664


In [8]:
ordenes = {n: orden(n)[0] for n in range(1, 1000)}
menor_de_orden = {}
for n, o in ordenes.items():
    menor_de_orden.setdefault(o, n)
print("Menor número de cada orden:", {o: menor_de_orden[o] for o in sorted(o for o in menor_de_orden if o)})
candidatos = [n for n, o in ordenes.items() if o is None]
print("Candidatos a Lychrel menores que 1000:", candidatos)

n = 196
for _ in range(1000):
    n = invierte_y_suma(n)
    assert not es_capicua(n)
print(f"196 tras 1000 pasos: {len(str(n))} cifras y ningún capicúa")
assert ordenes[89] == 24 and candidatos == [196, 295, 394, 493, 592, 689, 691, 788, 790, 879, 887, 978, 986]

Menor número de cada orden: {1: 1, 2: 5, 3: 59, 4: 69, 5: 166, 6: 79, 7: 188, 8: 193, 10: 829, 11: 167, 14: 849, 15: 177, 16: 999, 17: 739, 19: 989, 22: 869, 23: 187, 24: 89}
Candidatos a Lychrel menores que 1000: [196, 295, 394, 493, 592, 689, 691, 788, 790, 879, 887, 978, 986]
196 tras 1000 pasos: 411 cifras y ningún capicúa


## 5. Números abundantes

Un número es abundante si la suma de sus divisores propios lo supera (12: $1 + 2 + 3 + 4 + 6 = 16$), perfecto si la iguala y deficiente si se queda corta. Calcular la suma de divisores propios de todos los números hasta 20 000 con una criba, listar los abundantes menores que 100, contar cuántos hay de cada clase y responder: ¿son pares todos los abundantes? Encontrar el primer abundante impar y comprobar que todo múltiplo de un abundante también lo es.

In [9]:
LIMITE = 20_000
suma_divisores = [0] * (LIMITE + 1)
for d in range(1, LIMITE // 2 + 1):
    for m in range(2 * d, LIMITE + 1, d):
        suma_divisores[m] += d

abundantes = [n for n in range(1, LIMITE + 1) if suma_divisores[n] > n]
print("Abundantes < 100:", [n for n in abundantes if n < 100])
clases = Counter(
    "abundante" if suma_divisores[n] > n else "perfecto" if suma_divisores[n] == n else "deficiente"
    for n in range(1, LIMITE + 1)
)
print(f"Hasta {LIMITE}:", dict(clases))
impares = [n for n in abundantes if n % 2]
print("Primer abundante impar:", impares[0], "; impares hasta", LIMITE, ":", len(impares))

assert all(suma_divisores[n] == sum(d for d in range(1, n) if n % d == 0) for n in range(1, 1000))
assert impares[0] == 945 and clases["perfecto"] == 4
conjunto = set(abundantes)
assert all(m in conjunto for a in abundantes[:50] for m in range(a, LIMITE + 1, a))

Abundantes < 100: [12, 18, 20, 24, 30, 36, 40, 42, 48, 54, 56, 60, 66, 70, 72, 78, 80, 84, 88, 90, 96]
Hasta 20000: {'deficiente': 15043, 'perfecto': 4, 'abundante': 4953}
Primer abundante impar: 945 ; impares hasta 20000 : 43


## 6. Huecos entre primos y primos sexis

El hueco entre dos primos consecutivos es su diferencia. Con una criba hasta $10^6$, encontrar para cada longitud de hueco el primer primo tras el que aparece y comprobar los primeros con fuerza bruta. Después, listar las parejas de primos sexis menores que 100 (primos que se diferencian en 6, no necesariamente consecutivos) y los tríos $(p, p + 6, p + 12)$.

In [10]:
def criba(n):
    es = bytearray([1]) * (n + 1)
    es[0] = es[1] = 0
    for p in range(2, math.isqrt(n) + 1):
        if es[p]:
            es[p * p :: p] = bytes(len(range(p * p, n + 1, p)))
    return [i for i, v in enumerate(es) if v]


primos = criba(1_000_000)
primer_hueco = {}
for p, q in pairwise(primos):
    primer_hueco.setdefault(q - p, p)

print(f"{'hueco':>5} {'tras el primo':>14}")
for g in sorted(primer_hueco)[:20]:
    print(f"{g:>5} {primer_hueco[g]:>14}")
mayor = max(primer_hueco)
print(f"Mayor hueco por debajo de 10⁶: {mayor}, tras {primer_hueco[mayor]}")


def primer_hueco_bruto(g):
    p = 2
    while True:
        if es_primo(p) and es_primo(p + g) and not any(es_primo(m) for m in range(p + 1, p + g)):
            return p
        p += 1


assert all(primer_hueco_bruto(g) == primer_hueco[g] for g in [1, *range(2, 22, 2)])

hueco  tras el primo
    1              2
    2              3
    4              7
    6             23
    8             89
   10            139
   12            199
   14            113
   16           1831
   18            523
   20            887
   22           1129
   24           1669
   26           2477
   28           2971
   30           4297
   32           5591
   34           1327
   36           9551
   38          30593
Mayor hueco por debajo de 10⁶: 114, tras 492113


In [11]:
pocos = set(primos[:25])
sexis = [(p, p + 6) for p in sorted(pocos) if p + 6 in pocos and p + 6 < 100]
trios = [(p, p + 6, p + 12) for p in sorted(pocos) if {p + 6, p + 12} <= pocos and p + 12 < 100]
print("Parejas sexis:", sexis)
print("Tríos sexis:", trios)
assert sexis == [(p, p + 6) for p in range(2, 94) if es_primo(p) and es_primo(p + 6)]

Parejas sexis: [(5, 11), (7, 13), (11, 17), (13, 19), (17, 23), (23, 29), (31, 37), (37, 43), (41, 47), (47, 53), (53, 59), (61, 67), (67, 73), (73, 79), (83, 89)]
Tríos sexis: [(5, 11, 17), (7, 13, 19), (11, 17, 23), (17, 23, 29), (31, 37, 43), (41, 47, 53), (47, 53, 59), (61, 67, 73), (67, 73, 79)]


## 7. Juegos con bits

Escribir tres funciones con operadores de bits:

- `potencias_de_dos(n)`: descompone $n$ en potencias de dos distintas ($2026 = 1024 + 512 + 256 + 128 + 64 + 32 + 8 + 2$), aislando cada vez el bit a 1 más bajo con `n & -n`.
- `invierte_32(x)`: invierte el orden de los 32 bits de un entero sin signo.
- `suma_8(a, b)` y `resta_8(a, b)`: operan como un registro de 8 bits, quedándose con `& 0xFF` y devolviendo además si hubo acarreo (o préstamo).

Comprobarlas contra `bin`, el formateo con `:032b` y la aritmética módulo 256 con todas las parejas de bytes.

In [12]:
def potencias_de_dos(n):
    partes = []
    while n:
        bajo = n & -n
        partes.append(bajo)
        n ^= bajo
    return partes[::-1]


def invierte_32(x):
    r = 0
    for _ in range(32):
        r = (r << 1) | (x & 1)
        x >>= 1
    return r


def suma_8(a, b):
    s = a + b
    return s & 0xFF, s > 0xFF


def resta_8(a, b):
    return (a - b) & 0xFF, a < b


print("2026 =", " + ".join(map(str, potencias_de_dos(2026))))
x = 0b1011_0000_0000_0000_0000_0000_0000_0110
print(f"{x:032b}\n{invierte_32(x):032b}")
for a, b in [(200, 100), (255, 1), (17, 25)]:
    print(f"{a} + {b} = {suma_8(a, b)}   {a} - {b} = {resta_8(a, b)}")

2026 = 1024 + 512 + 256 + 128 + 64 + 32 + 8 + 2
10110000000000000000000000000110
01100000000000000000000000001101
200 + 100 = (44, True)   200 - 100 = (100, False)
255 + 1 = (0, True)   255 - 1 = (254, False)
17 + 25 = (42, False)   17 - 25 = (248, True)


In [13]:
for n in range(1, 5000):
    partes = potencias_de_dos(n)
    assert sum(partes) == n and len(partes) == bin(n).count("1") == len(set(partes))
random.seed(5)
for x in [0, 1, 2**32 - 1] + [random.getrandbits(32) for _ in range(1000)]:
    assert invierte_32(x) == int(f"{x:032b}"[::-1], 2) and invierte_32(invierte_32(x)) == x
assert all(suma_8(a, b) == ((a + b) % 256, a + b >= 256) and resta_8(a, b) == ((a - b) % 256, a < b)
           for a in range(256) for b in range(256))

## 8. Variaciones y combinaciones sin factoriales

Calcular $V(n, k) = n (n - 1) \cdots (n - k + 1)$ y $C(n, k) = \binom{n}{k}$ sin calcular factoriales completos: las variaciones con un producto de $k$ factores y las combinaciones multiplicando y dividiendo paso a paso, $C \leftarrow C \cdot (n - k + i) / i$ (cada paso da un entero, el número combinatorio $\binom{n-k+i}{i}$), usando la simetría $\binom{n}{k} = \binom{n}{n-k}$ para hacer como mucho $n/2$ pasos. Comparar con `math.perm` y `math.comb`, y el tamaño del mayor número intermedio frente a la fórmula con factoriales.

In [14]:
def variaciones(n, k):
    if not 0 <= k <= n:
        return 0
    r = 1
    for f in range(n - k + 1, n + 1):
        r *= f
    return r


def combinaciones(n, k, intermedios=None):
    if not 0 <= k <= n:
        return 0
    k = min(k, n - k)
    r = 1
    for i in range(1, k + 1):
        producto = r * (n - k + i)
        if intermedios is not None:
            intermedios.append(producto)
        r = producto // i
    return r


assert all(variaciones(n, k) == math.perm(n, k) and combinaciones(n, k) == math.comb(n, k)
           for n in range(80) for k in range(n + 2))

print(f"{'n':>7} {'k':>6} {'cifras C(n,k)':>14} {'mayor intermedio':>17} {'cifras de n!':>13}")
for n, k in [(49, 6), (100, 3), (1000, 3), (1000, 500), (10**6, 999_998)]:
    intermedios = []
    c = combinaciones(n, k, intermedios)
    cifras_fact = len(str(math.factorial(n))) if n <= 1000 else "—"
    print(f"{n:>7} {k:>6} {len(str(c)):>14} {len(str(max(intermedios))):>17} {cifras_fact:>13}")
print("C(10⁶, 999998) =", combinaciones(10**6, 999_998), "en 2 pasos")

      n      k  cifras C(n,k)  mayor intermedio  cifras de n!
     49      6              8                 8            63
    100      3              6                 6           158
   1000      3              9                 9          2568
   1000    500            300               303          2568
1000000 999998             12                12             —
C(10⁶, 999998) = 499999500000 en 2 pasos


## 9. Ternas pitagóricas primitivas

Una terna $a^2 + b^2 = c^2$ es primitiva si $\gcd(a, b, c) = 1$. La fórmula de Euclides las da todas: $a = m^2 - n^2$, $b = 2mn$, $c = m^2 + n^2$ con $m > n > 0$ primos entre sí y de distinta paridad. Generar las de $c \le 100$, comprobarlas con fuerza bruta hasta $c \le 1000$ y verificar las propiedades: exactamente un cateto es par, un cateto es múltiplo de 3, uno de los tres números es múltiplo de 5 y la hipotenusa es impar.

In [15]:
def ternas_euclides(c_max):
    ternas = []
    for m in range(2, math.isqrt(c_max) + 1):
        for n in range(1, m):
            if (m - n) % 2 and math.gcd(m, n) == 1 and m * m + n * n <= c_max:
                a, b = m * m - n * n, 2 * m * n
                ternas.append((min(a, b), max(a, b), m * m + n * n))
    return sorted(ternas, key=lambda t: (t[2], t[0]))


def ternas_bruta(c_max):
    ternas = []
    for c in range(1, c_max + 1):
        for a in range(1, c):
            b = math.isqrt(c * c - a * a)
            if a < b and a * a + b * b == c * c and math.gcd(a, b) == 1:
                ternas.append((a, b, c))
    return ternas


print(ternas_euclides(100))
ternas = ternas_euclides(1000)
assert ternas == ternas_bruta(1000)
for a, b, c in ternas:
    assert (a % 2 == 0) + (b % 2 == 0) == 1 and c % 2 == 1
    assert a % 3 == 0 or b % 3 == 0
    assert 0 in (a % 5, b % 5, c % 5)
print(f"{len(ternas)} ternas primitivas con c ≤ 1000; todas cumplen las propiedades")

[(3, 4, 5), (5, 12, 13), (8, 15, 17), (7, 24, 25), (20, 21, 29), (12, 35, 37), (9, 40, 41), (28, 45, 53), (11, 60, 61), (16, 63, 65), (33, 56, 65), (48, 55, 73), (13, 84, 85), (36, 77, 85), (39, 80, 89), (65, 72, 97)]


158 ternas primitivas con c ≤ 1000; todas cumplen las propiedades

## 10. Sucesiones de Recamán y de Padovan

- **Recamán**: $a_0 = 0$ y $a_n = a_{n-1} - n$ si ese valor es positivo y no ha salido aún; si no, $a_n = a_{n-1} + n$. Mostrar los primeros términos y encontrar el primer valor que se repite.
- **Padovan**: $P_0 = P_1 = P_2 = 1$ y $P_n = P_{n-2} + P_{n-3}$. Comprobar la identidad $P_n = P_{n-1} + P_{n-5}$ y que el cociente $P_{n+1} / P_n$ tiende al número plástico, la raíz real de $x^3 = x + 1$.

In [16]:
def recaman(cuantos):
    terminos, vistos = [0], {0}
    for n in range(1, cuantos):
        a = terminos[-1] - n
        if a <= 0 or a in vistos:
            a = terminos[-1] + n
        terminos.append(a)
        vistos.add(a)
    return terminos


r = recaman(1000)
print("Recamán:", r[:25])
repetido = next(n for n in range(len(r)) if r[n] in r[:n])
print(f"Primera repetición: a_{repetido} = {r[repetido]} = a_{r.index(r[repetido])}")
assert all(abs(b - a) == n for n, (a, b) in enumerate(pairwise(r), start=1))

Recamán: [0, 1, 3, 6, 2, 7, 13, 20, 12, 21, 11, 22, 10, 23, 9, 24, 8, 25, 43, 62, 42, 63, 41, 18, 42]
Primera repetición: a_24 = 42 = a_20


In [17]:
def padovan(cuantos):
    p = [1, 1, 1]
    while len(p) < cuantos:
        p.append(p[-2] + p[-3])
    return p[:cuantos]


p = padovan(120)
print("Padovan:", p[:20])
assert all(p[n] == p[n - 1] + p[n - 5] for n in range(5, len(p)))
plastico = p[-1] / p[-2]
print(f"P₁₂₀ / P₁₁₉ = {plastico:.12f}; x³ - x - 1 = {plastico**3 - plastico - 1:.1e}")
assert abs(plastico**3 - plastico - 1) < 1e-12

Padovan: [1, 1, 1, 2, 2, 3, 4, 5, 7, 9, 12, 16, 21, 28, 37, 49, 65, 86, 114, 151]
P₁₂₀ / P₁₁₉ = 1.324717957245; x³ - x - 1 = 2.2e-16


## 11. Sucesiones de Golomb y de Ulam

- **Golomb**: sucesión no decreciente en la que cada $n$ aparece exactamente $a_n$ veces (1, 2, 2, 3, 3, 4, 4, 4…). Se calcula con $a_1 = 1$, $a_n = 1 + a_{n - a_{a_{n-1}}}$. Comprobar que se describe a sí misma.
- **Ulam**: empieza por 1 y 2; cada término siguiente es el menor entero mayor que el último que se escribe de una única forma como suma de dos términos anteriores distintos. Calcularla llevando la cuenta de las sumas de parejas y comprobarla contra la definición.

In [18]:
def golomb(cuantos):
    a = [0, 1]
    for n in range(2, cuantos + 1):
        a.append(1 + a[n - a[a[n - 1]]])
    return a[1:]


g = golomb(5000)
print("Golomb:", g[:25])
apariciones = Counter(g)
assert all(apariciones[n] == g[n - 1] for n in range(1, g[-1]))
assert g == sorted(g)

Golomb: [1, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8, 9, 9]


In [19]:
def ulam(cuantos):
    terminos = [1, 2]
    sumas = Counter({3: 1})
    while len(terminos) < cuantos:
        siguiente = next(s for s in range(terminos[-1] + 1, 2 * terminos[-1]) if sumas[s] == 1)
        sumas.update(siguiente + t for t in terminos)
        terminos.append(siguiente)
    return terminos


def ulam_definicion(cuantos):
    terminos = [1, 2]
    while len(terminos) < cuantos:
        s = terminos[-1] + 1
        while sum(1 for i, x in enumerate(terminos) for y in terminos[i + 1 :] if x + y == s) != 1:
            s += 1
        terminos.append(s)
    return terminos


u = ulam(300)
print("Ulam:", u[:25])
assert u[:60] == ulam_definicion(60)

Ulam: [1, 2, 3, 4, 6, 8, 11, 13, 16, 18, 26, 28, 36, 38, 47, 48, 53, 57, 62, 69, 72, 77, 82, 87, 97]


## 12. Problema de Josefo

$n$ personas en círculo, numeradas desde 1, y se elimina a una de cada $k$ hasta que queda una. Resolverlo de tres formas y comprobar que coinciden:

- Simulando el círculo con un `deque` y `rotate`, lo que da además el orden de eliminación.
- Con la recurrencia $J(1) = 0$, $J(n) = (J(n - 1) + k) \bmod n$ (posiciones desde 0).
- Para $k = 2$, con la fórmula cerrada: si $n = 2^m + L$ con $0 \le L < 2^m$, sobrevive $2L + 1$, que equivale a pasar el bit más alto de $n$ al final.

In [20]:
def josefo_simulado(n, k):
    circulo, orden = deque(range(1, n + 1)), []
    while circulo:
        circulo.rotate(-(k - 1))
        orden.append(circulo.popleft())
    return orden


def josefo_recurrencia(n, k):
    j = 0
    for m in range(2, n + 1):
        j = (j + k) % m
    return j + 1


def josefo_k2(n):
    alto = 1 << (n.bit_length() - 1)
    return 2 * (n - alto) + 1


print("Orden de eliminación con n = 7, k = 3:", josefo_simulado(7, 3))
print(f"{'n':>3} {'binario':>8} {'sobrevive (k=2)':>16} {'binario':>8}")
for n in (1, 5, 6, 7, 8, 13, 41):
    print(f"{n:>3} {n:>8b} {josefo_k2(n):>16} {josefo_k2(n):>8b}")

assert all(josefo_simulado(n, k)[-1] == josefo_recurrencia(n, k) for n in range(1, 151) for k in range(1, 8))
assert all(josefo_k2(n) == josefo_recurrencia(n, 2) == int(f"{n:b}"[1:] + "1", 2) for n in range(2, 5000))
print("Con 10⁶ personas y k = 3 sobrevive la", josefo_recurrencia(10**6, 3))

Orden de eliminación con n = 7, k = 3: [3, 6, 2, 7, 5, 1, 4]
  n  binario  sobrevive (k=2)  binario
  1        1                1        1
  5      101                3       11
  6      110                5      101
  7      111                7      111
  8     1000                1        1
 13     1101               11     1011
 41   101001               19    10011


Con 10⁶ personas y k = 3 sobrevive la 637798


## 13. Siguiente permutación de las cifras

Dado un número, encontrar el menor número mayor que él formado con las mismas cifras (o `None` si no existe): buscar desde la derecha la primera cifra menor que su vecina derecha, cambiarla por la menor cifra mayor que ella de su derecha e invertir el tramo final. Comprobarlo con `itertools.permutations` y verificar que, aplicándolo desde la menor ordenación de unas cifras, se recorren todas sus permutaciones distintas.

In [21]:
def siguiente_permutacion(n):
    c = list(str(n))
    i = len(c) - 2
    while i >= 0 and c[i] >= c[i + 1]:
        i -= 1
    if i < 0:
        return None
    j = len(c) - 1
    while c[j] <= c[i]:
        j -= 1
    c[i], c[j] = c[j], c[i]
    c[i + 1 :] = reversed(c[i + 1 :])
    return int("".join(c))


def siguiente_bruta(n):
    mayores = [int("".join(p)) for p in permutations(str(n))]
    return min((m for m in mayores if m > n), default=None)


for n in (2026, 38276, 115, 1999, 4321):
    print(f"{n:>6} → {siguiente_permutacion(n)}")

random.seed(9)
for n in [random.randrange(1, 10**6) for _ in range(300)]:
    assert siguiente_permutacion(n) == siguiente_bruta(n)

recorrido, n = [], 1123
while n is not None:
    recorrido.append(n)
    n = siguiente_permutacion(n)
print(len(recorrido), "permutaciones de 1123:", recorrido)
assert recorrido == sorted({int("".join(p)) for p in permutations("1123")})

  2026 → 2062
 38276 → 38627
   115 → 151
  1999 → 9199
  4321 → None
12 permutaciones de 1123: [1123, 1132, 1213, 1231, 1312, 1321, 2113, 2131, 2311, 3112, 3121, 3211]


## 14. Secuencia de Farey

La secuencia de Farey de orden $n$ son las fracciones irreducibles de $[0, 1]$ con denominador hasta $n$, ordenadas. Generarla de dos formas: con un conjunto de `Fraction` ordenado, y término a término sabiendo que, si $a/b$ y $c/d$ son vecinas, la siguiente es $(kc - a)/(kd - b)$ con $k = \lfloor (n + b)/d \rfloor$. Comprobar que los vecinos cumplen $bc - ad = 1$, que cada término es la mediante $\frac{a+c}{b+d}$ de sus vecinos, y que su longitud es $1 + \sum_{k=1}^{n} \varphi(k)$, con $\varphi$ la función de Euler.

In [22]:
def farey_conjunto(n):
    return sorted({Fraction(a, b) for b in range(1, n + 1) for a in range(b + 1)})


def farey_vecinos(n):
    a, b, c, d = 0, 1, 1, n
    terminos = [Fraction(a, b)]
    while c <= n:
        k = (n + b) // d
        a, b, c, d = c, d, k * c - a, k * d - b
        terminos.append(Fraction(a, b))
    return terminos


def phi(n):
    resultado, p = n, 2
    while p * p <= n:
        if n % p == 0:
            while n % p == 0:
                n //= p
            resultado -= resultado // p
        p += 1
    return resultado - resultado // n if n > 1 else resultado


print("F₅:", " ".join(str(f) for f in farey_vecinos(5)))
for n in range(1, 61):
    f = farey_vecinos(n)
    assert f == farey_conjunto(n)
    assert all(x.denominator * y.numerator - x.numerator * y.denominator == 1 for x, y in pairwise(f))
    assert all(Fraction(x.numerator + z.numerator, x.denominator + z.denominator) == y
               for x, y, z in zip(f, f[1:], f[2:]))
    assert len(f) == 1 + sum(phi(k) for k in range(1, n + 1))
assert [phi(n) for n in range(1, 13)] == [sum(math.gcd(n, k) == 1 for k in range(1, n + 1)) for n in range(1, 13)]
print(f"{'n':>4} {'|Fₙ|':>6}")
for n in (1, 5, 10, 100, 1000):
    print(f"{n:>4} {1 + sum(phi(k) for k in range(1, n + 1)):>6}")

F₅: 0 1/5 1/4 1/3 2/5 1/2 3/5 2/3 3/4 4/5 1
   n   |Fₙ|
   1      2
   5     11
  10     33
 100   3045
1000 304193


## 15. Dígitos de π con enteros grandes

La fórmula de Machin, $\frac{\pi}{4} = 4 \arctan \frac{1}{5} - \arctan \frac{1}{239}$, converge deprisa porque $\arctan \frac{1}{x} = \frac{1}{x} - \frac{1}{3x^3} + \frac{1}{5x^5} - \cdots$. Calcular $d$ decimales de π solo con enteros: multiplicar todo por $10^{d + 10}$ (diez cifras de guarda contra los redondeos de la división entera) y sumar los términos hasta que valgan 0. Comparar los 100 primeros decimales con una constante conocida y los 1000 primeros con `mpmath`, y contar cuántas veces aparece cada cifra.

In [23]:
import mpmath


def arctan_inverso(x, escala):
    potencia = escala // x
    total, k = potencia, 0
    while potencia:
        potencia //= x * x
        k += 1
        total += (-1) ** k * (potencia // (2 * k + 1))
    return total


def digitos_pi(d):
    escala = 10 ** (d + 10)
    pi = 4 * (4 * arctan_inverso(5, escala) - arctan_inverso(239, escala))
    s = str(pi // 10**10)
    return s[0] + "." + s[1:]


PI_100 = ("3.1415926535897932384626433832795028841971693993751058209749445923078164062862089986280348253421170679")
assert digitos_pi(100) == PI_100
mpmath.mp.dps = 1020
pi_1000 = digitos_pi(1000)
assert pi_1000 == mpmath.nstr(mpmath.pi, 1015)[:1002]
print(pi_1000[:62] + "…")
print("Frecuencia de cada cifra en los 1000 decimales:", dict(sorted(Counter(pi_1000[2:]).items())))

3.141592653589793238462643383279502884197169399375105820974944…
Frecuencia de cada cifra en los 1000 decimales: {'0': 93, '1': 116, '2': 103, '3': 102, '4': 93, '5': 97, '6': 94, '7': 95, '8': 101, '9': 106}
